<a href="https://colab.research.google.com/github/laurakeita/MMM/blob/main/notebooks/02_train_meridian_mmm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Train Meridian MMM
Bayesian MMM training with log-normal ROI priors.

In [ ]:
!pip install git+https://github.com/google/meridian.git altair jsonschema==3.2.0 -q

In [ ]:
!pip install jsonschema==3.2.0
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
import arviz as az

import IPython
import jsonschema
import importlib

# Reload jsonschema to ensure the newly installed version is active
importlib.reload(jsonschema)

from meridian import constants
from meridian.data import load
from meridian.data import test_utils
from meridian.model import model
from meridian.model import spec
from meridian.model import prior_distribution
from meridian.analysis import optimizer
from meridian.analysis import analyzer
from meridian.analysis import visualizer
from meridian.analysis import summarizer
from meridian.analysis import formatter

# check if GPU is available
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print("Num CPUs Available: ", len(tf.config.experimental.list_physical_devices('CPU')))

## Mount Drive & Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

*Run 01_data_audit first to define `data`, `media`, `impressions`, `output`, `cost_mapping`, `impressions_mapping`.*

## Build Meridian Dataset

In [ ]:
from meridian.data import load
import numpy as np # Import numpy for array comparison

print(data.columns)

# Detect controls dynamically
controls_candidates = ['Offline Discount']
controls = [c for c in controls_candidates if c in data.columns]

coord_to_columns = load.CoordToColumns(
    time='date',
    kpi=output[0],
    controls=controls if controls else None,
    media=impressions, # Pass the list of impression columns
    media_spend=media, # Pass the list of spend columns
)

# Add a check to compare impressions list and impressions_mapping keys
print("\nComparing impressions list and impressions_mapping keys:")
print("Impressions list:", impressions)
print("Impressions mapping keys:", list(impressions_mapping.keys()))

# Check if keys match the impressions list
if np.array_equal(sorted(impressions), sorted(list(impressions_mapping.keys()))):
    print("Impressions list and impressions_mapping keys match.")
else:
    print("Impressions list and impressions_mapping keys DO NOT match.")


loader = load.DataFrameDataLoader(
    df=data, # Use the data variable
    kpi_type='revenue',
    coord_to_columns=coord_to_columns,
    media_spend_to_channel=cost_mapping,
    media_to_channel=impressions_mapping, # Add the impressions mapping
)
data_meridian = loader.load()

In [ ]:
data_meridian.as_dataset()

## Define Priors

In [ ]:
#functiom to translate the media ROI into a log normal distribution usable by the model prios

def estimate_lognormal_dist(mean, std):
    """
    Reparameterizes the LogNormal distribution in terms of its mean and std.
    Returns mu_log and std_log which can be used to define a LogNormal.
    """
    mu_log = np.log(mean) - 0.5 * np.log((std/mean)**2 + 1)
    std_log = np.sqrt(np.log((std/mean)**2 + 1))
    return mu_log, std_log

roi_mu= 2
roi_sigma= 1.5

roi_mu_log, roi_sigma_log=estimate_lognormal_dist(roi_mu, roi_sigma)
print(roi_mu_log, roi_sigma_log)
"""
# setting priors based on spend share
total_spend= data[media].sum().sum()
priors={'mu':[], 'sigma':[]}
for i in media:
  spend_share=(data[i].sum()/total_spend)*2
  roi_sigma=roi_mu*spend_share
  mu , sigma= estimate_lognormal_dist(roi_mu, roi_sigma)
  priors['mu'].append(mu)
  priors['sigma'].append(sigma)

print(priors)

roi_mu=priors['mu']
roi_sigma=priors['sigma']
print(roi_mu)"""


## Build Model Spec

In [ ]:
# Prior Distribution
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
       loc=np.float32(roi_mu_log),
       scale=np.float32(roi_sigma_log),
       name=constants.ROI_M)
)

# Build Meridian Model
# Initializing ModelSpec directly with parameters from the old create_model_spec call and the prior
# Mapping n_lags and adstock_max_lag_weight to max_lag (assuming this is the correct parameter name)
model_spec = spec.ModelSpec(
    prior=prior,
    max_lag=7, # Assuming adstock_max_lag_weight maps to max_lag
)

# Initialize the model for posterior sampling
mmm = model.Meridian(input_data=data_meridian, model_spec=model_spec)

## Train (Sample Posterior)

In [ ]:
from tqdm import tqdm
mmm.sample_prior(100)

mmm.sample_posterior(
    n_chains=5,
    n_adapt=1000,
    n_burnin=1000,
    n_keep=2000,
)

model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.predictive_accuracy_table()

